In [1]:
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# ----------------- config -----------------

BASE = "https://www.wikiart.org"
TEXT_LIST_TMPL = BASE + "/en/{slug}/all-works/text-list"

# local_folder_name -> wikiart slug
ARTISTS = {
    "Joan_Miro": "joan-miro",
    "Salvador_Dali": "salvador-dali",
    "Rene_Magritte": "rene-magritte",
    "Pablo_Picasso": "pablo-picasso",
    "Jackson_Pollock": "jackson-pollock",
    "Rembrandt": "rembrandt",
    "Caravaggio": "caravaggio",
    "Camille_Pissarro": "camille-pissarro",
    "Alfred_Sisley": "alfred-sisley",
    "Claude_Monet": "claude-monet",
    "Vincent_van_Gogh": "vincent-van-gogh",
}

IMAGES_PER_ARTIST = 50
OUTPUT_ROOT = Path("data")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Heri-art-downloader/1.0)",
}


# ----------------- helpers -----------------

def fetch_html(url: str) -> str:
    """GET a page and return text (raises on HTTP errors)."""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.text


def get_painting_urls(slug: str, limit: int) -> list[str]:
    """
    From the 'all-works/text-list' page for a given artist slug,
    extract up to 'limit' URLs of individual painting pages.
    """
    url = TEXT_LIST_TMPL.format(slug=slug)
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    urls = []
    seen = set()

    # On that page, each work is listed as a bullet with a link.
    for a in soup.find_all("a"):
        href = a.get("href", "")
        # only artwork links for this artist
        if not href:
            continue
        if not href.startswith("/en/"):
            continue
        if f"/{slug}/" not in href:
            continue

        full = urljoin(BASE, href)
        if full in seen:
            continue

        seen.add(full)
        urls.append(full)

        if len(urls) >= limit:
            break

    return urls


def get_image_url(painting_url: str) -> str | None:
    """
    From a painting page, extract the direct image URL.
    On WikiArt pages it is usually a link to uploadsX.wikiart.org.
    """
    html = fetch_html(painting_url)
    soup = BeautifulSoup(html, "html.parser")

    # Look for the first <a> that points to an uploads*.wikiart.org image
    candidate = soup.find(
        "a",
        href=lambda h: h and "uploads" in h and "wikiart.org" in h
    )
    if not candidate:
        return None

    href = candidate.get("href")
    if not href:
        return None

    return urljoin(painting_url, href)


def download_image(url: str, dest: Path) -> None:
    """Download 'url' to file 'dest'."""
    resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
    resp.raise_for_status()
    dest.parent.mkdir(parents=True, exist_ok=True)

    with dest.open("wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

In [3]:
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# ------------- config -------------

BASE = "https://www.wikiart.org"
TEXT_LIST_TMPL = BASE + "/en/{slug}/all-works/text-list"

# local_folder_name -> wikiart slug
ARTISTS = {
    "Joan_Miro": "joan-miro",
    "Salvador_Dali": "salvador-dali",
    "Rene_Magritte": "rene-magritte",
    "Pablo_Picasso": "pablo-picasso",
    "Jackson_Pollock": "jackson-pollock",
    "Rembrandt": "rembrandt",
    "Caravaggio": "caravaggio",
    "Camille_Pissarro": "camille-pissarro",
    "Alfred_Sisley": "alfred-sisley",
    "Claude_Monet": "claude-monet",
    "Vincent_van_Gogh": "vincent-van-gogh",
}

IMAGES_PER_ARTIST = 50
OUTPUT_ROOT = Path("data")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Heri-art-downloader/1.0)",
}


# ------------- helpers -------------

def fetch_html(url: str) -> str:
    """GET a page and return its text, raising on HTTP errors."""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.text


def get_painting_urls(slug: str, limit: int) -> list[str]:
    """
    From the 'all-works/text-list' page for a given artist slug,
    extract up to 'limit' URLs of individual painting pages.
    """
    url = TEXT_LIST_TMPL.format(slug=slug)
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    urls: list[str] = []
    seen: set[str] = set()

    # Each work appears as an <a> linking to '/en/<slug>/<title>'
    for a in soup.find_all("a"):
        href = a.get("href", "")
        if not href:
            continue
        if not href.startswith("/en/"):
            continue
        if f"/{slug}/" not in href:
            continue

        full = urljoin(BASE, href)
        if full in seen:
            continue
        seen.add(full)
        urls.append(full)

        if len(urls) >= limit:
            break

    return urls


def get_image_url(painting_url: str) -> str | None:
    """
    From a painting page, extract the direct image URL.

    Primary method:
      - read <meta property="og:image" content="...">
      - strip any size suffix after '!' (e.g. '!Large.jpg')

    Fallback:
      - first <a> whose href contains 'uploads' and 'wikiart.org'
    """
    html = fetch_html(painting_url)
    soup = BeautifulSoup(html, "html.parser")

    # 1) og:image
    og = soup.find("meta", attrs={"property": "og:image"})
    if og is not None:
        content = og.get("content")
        if content:
            # often looks like ...jpg!Large.jpg
            return content.split("!")[0]

    # 2) fallback: <a href="https://uploads...wikiart.org/...jpg">
    a = soup.find("a", href=lambda h: h and "uploads" in h and "wikiart.org" in h)
    if a is not None:
        href = a.get("href")
        if href:
            return urljoin(painting_url, href)

    return None


def download_image(url: str, dest: Path) -> bool:
    """
    Download 'url' to file 'dest'.

    Returns True if a valid image was saved, False otherwise.
    """
    resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
    # Some sites return 200 with HTML error pages; check content-type.
    ct = resp.headers.get("Content-Type", "")
    if not ct.startswith("image/"):
        print(f"  non-image content-type '{ct}' from {url}, skipping")
        return False

    dest.parent.mkdir(parents=True, exist_ok=True)
    with dest.open("wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    return True


# ------------- main routine -------------

def main() -> None:
    OUTPUT_ROOT.mkdir(exist_ok=True)

    for folder_name, slug in ARTISTS.items():
        painter_dir = OUTPUT_ROOT / folder_name
        painter_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n=== {folder_name} (slug: {slug}) ===")

        # existing images (so you can resume)
        existing = sorted(painter_dir.glob("*.jpg"))
        start_idx = len(existing) + 1

        if start_idx > IMAGES_PER_ARTIST:
            print("Already have enough images, skipping.")
            continue

        needed = IMAGES_PER_ARTIST - (start_idx - 1)
        print(f"Already have {start_idx - 1}, will try to fetch {needed} more.")

        painting_urls = get_painting_urls(slug, limit=IMAGES_PER_ARTIST * 2)
        print(f"Found {len(painting_urls)} painting pages.")

        count = 0
        for i, p_url in enumerate(painting_urls, start=1):
            if start_idx + count > IMAGES_PER_ARTIST:
                break

            print(f"[{i}/{len(painting_urls)}] page: {p_url}")

            try:
                img_url = get_image_url(p_url)
            except Exception as e:
                print(f"  error fetching image url: {e}")
                continue

            if not img_url:
                print("  no image url found")
                continue

            idx = start_idx + count
            out_path = painter_dir / f"{folder_name}_{idx:03d}.jpg"
            print(f"  -> {img_url}")
            print(f"  saving as {out_path}")

            try:
                ok = download_image(img_url, out_path)
            except Exception as e:
                print(f"  error downloading image: {e}")
                ok = False

            if ok:
                count += 1
            else:
                # if a bad HTML file was written accidentally, remove it
                if out_path.exists() and out_path.stat().st_size < 5000:
                    out_path.unlink(missing_ok=True)

            # be polite with the server
            time.sleep(1.0)

        print(f"Downloaded {count} new images for {folder_name}.")


if __name__ == "__main__":
    # Check WikiArt terms of use yourself.
    # Use this only in allowed research / educational settings.
    main()



=== Joan_Miro (slug: joan-miro) ===
Already have 0, will try to fetch 50 more.
Found 100 painting pages.
[1/100] page: https://www.wikiart.org/en/joan-miro/the-farmer
  -> https://uploads0.wikiart.org/images/joan-miro/the-farmer.jpg
  saving as data\Joan_Miro\Joan_Miro_001.jpg
[2/100] page: https://www.wikiart.org/en/joan-miro/portrait-of-a-young-girl
  -> https://uploads5.wikiart.org/images/joan-miro/portrait-of-a-young-girl.jpg
  saving as data\Joan_Miro\Joan_Miro_002.jpg
[3/100] page: https://www.wikiart.org/en/joan-miro/not_detected_227961
  -> https://uploads0.wikiart.org/images/joan-miro/not_detected_227961.jpg
  saving as data\Joan_Miro\Joan_Miro_003.jpg
[4/100] page: https://www.wikiart.org/en/joan-miro/still-life-with-rose
  -> https://uploads1.wikiart.org/images/joan-miro/still-life-with-rose.jpg
  saving as data\Joan_Miro\Joan_Miro_004.jpg
[5/100] page: https://www.wikiart.org/en/joan-miro/ciurana-the-path
  -> https://uploads5.wikiart.org/images/joan-miro/ciurana-the-path.